# Scalar KRR experiments — Section 6.1, Figure 1

Noiseless kernel ridge regression on $\mathcal{X}=[-1,1]^P$ ($P\in\{1,5\}$, uniform
measure) for the two analytic targets $f_{*,1}(x)=e^{-\|x\|^2}$ and
$f_{*,2}(x)=\|x\|^2$.

## Kernels and grids

The kernels are implemented directly in the normalisation of the paper, so that the
parameters below **are** the $\bar s$ and $c$ of eqs. (22), (24) and (1):

$$k_{\mathrm{Gauss}}(x,x')=\exp(-\bar s^{\,2}\|x-x'\|^2),\qquad
k_{\mathrm{Sob}}(x,x')=\frac{2^{1-\bar r}}{\Gamma(\bar r)}
\bigl(\sqrt{2\bar r}\,\bar s d\bigr)^{\bar r}\mathcal{K}_{\bar r}
\bigl(\sqrt{2\bar r}\,\bar s d\bigr),\qquad
k_{\mathrm{Szeg}}(x,x')=\prod_{p=1}^{P}\frac{1}{1-c\,x_px_p'} .$$

For $\bar r=3/2$ and $5/2$ the Matérn kernel has the closed forms
$(1+d)e^{-d}$ and $(1+d+d^2/3)e^{-d}$ with $d=\sqrt{2\bar r}\,\bar s\|x-x'\|$.
Notebook 03 uses the same expressions, so the two experiments share one kernel
implementation as well as one grid.

`S_GRID` and `C_GRID` below are the **shared** grids of eq. (25); the parametric-PDE
experiment of Section 6.2 uses exactly the same two. They span five decades, and for
$N\geq700$ the cross-validated choice is interior for every kernel and every panel.


## Protocol

* Model selection: 3-fold CV over kernel parameter × regularization
  $\lambda\in\{10^{-3},10^{-5},10^{-7},10^{-9}\}$, added **directly to the kernel
  matrix**. Non-finite kernel matrices / failed solves are skipped.
* $N\in\{10,\ldots,1000\}$, 10 repetitions with deterministic seeds, 20000 test
  points per repetition, float64 throughout.
* Reported: **median and interquartile range** over the repetitions. 
* The selected hyperparameters are stored alongside the errors (`sel_*`), so the
  boundary check above can be reproduced.

Outputs: `scalar_P{P}_{f}.npz` and the four panels of Figure 1,
`P={P}-f={f}-krr-tube{FIG_SUFFIX}.pdf`.

Runtime: about 5 h in total on one CPU ($P=5$ dominates).

In [ ]:
import numpy as np
from sklearn.model_selection import KFold

Ns = [10, 30, 50, 70, 100, 200, 300, 400, 500, 600, 700, 800, 900, 1000]
REPEATS = 10
N_TEST = 20000

# ---- shared hyper-parameter grids, eq. (25); identical to notebook 03 ----
S_GRID = [0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100]      # sbar, eqs. (22)/(24)
C_GRID = [0.01, 0.02, 0.05, 0.1, 0.3, 0.5, 0.7, 0.9, 0.99]     # Szego parameter c
ALPHAS = [1e-3, 1e-5, 1e-7, 1e-9]        # nugget, added directly to K

# The paper includes the four panels as "P={P}-f={f}-krr-tube_new.pdf".
# Set FIG_SUFFIX = "" if you prefer plain file names (and adapt the .tex accordingly).
FIG_SUFFIX = "_new"


def f_target(f_type, x):
    """The two analytic targets of Section 6.1."""
    return (np.linalg.norm(x, axis=1) ** 2 if f_type == "poly"
            else np.exp(-np.linalg.norm(x, axis=1) ** 2))


def _sqdist(X, Y):
    return np.maximum(np.sum(X**2, 1)[:, None] + np.sum(Y**2, 1)[None, :]
                      - 2 * X @ Y.T, 0.0)


def gauss(X, Y, s):
    """Paper eq. (22): exp(-sbar^2 |x-x'|^2)."""
    return np.exp(-(s**2) * _sqdist(X, Y))


def matern(X, Y, s, rbar):
    """Paper eq. (24): argument sqrt(2 rbar) * sbar * |x-x'|."""
    d = np.sqrt(_sqdist(X, Y)) * s * np.sqrt(2.0 * rbar)
    return (1 + d) * np.exp(-d) if rbar == 1.5 else (1 + d + d**2 / 3) * np.exp(-d)


def szego_prod(X, Y, c):
    """Product Szego kernel prod_p 1/(1 - c x_p y_p)  (positive definite for c<1)."""
    return np.prod(1.0 / (1.0 - c * X[:, None, :] * Y[None, :, :]), axis=2)


KERNELS = {
    "Matern_1.5": [(s, lambda A, B, s=s: matern(A, B, s, 1.5)) for s in S_GRID],
    "Matern_2.5": [(s, lambda A, B, s=s: matern(A, B, s, 2.5)) for s in S_GRID],
    "RBF":        [(s, lambda A, B, s=s: gauss(A, B, s))       for s in S_GRID],
    "Szego_prod": [(c, lambda A, B, c=c: szego_prod(A, B, c))  for c in C_GRID],
}


def krr(K, y, a):
    """KRR coefficients: (K + a I)^{-1} y."""
    return np.linalg.solve(K + a * np.eye(K.shape[0]), y)

In [ ]:
def run_scalar(P):
    """Full experiment for one input dimension P; writes scalar_P{P}_{f}.npz.

    Seeds are deterministic per (N, repetition), so results are exactly
    reproducible: rng = default_rng(10000 * Ns.index(N) + rep). Besides the errors,
    the selected (kernel parameter, nugget) pair is stored for every repetition."""
    for f_type in ["poly", "exp"]:
        results, chosen = {k: {} for k in KERNELS}, {k: {} for k in KERNELS}
        for N in Ns:
            errs = {k: [] for k in KERNELS}
            sel = {k: [] for k in KERNELS}
            for rep in range(REPEATS):
                rng = np.random.default_rng(10000 * Ns.index(N) + rep)
                X = rng.uniform(-1, 1, (N, P)); y = f_target(f_type, X)
                X_te = rng.uniform(-1, 1, (N_TEST, P)); y_te = f_target(f_type, X_te)
                kf = list(KFold(3, shuffle=True, random_state=0).split(X))
                for kname, grid in KERNELS.items():
                    best, bv = None, np.inf
                    for pval, kfun in grid:          # CV: kernel parameter x nugget
                        K_full = kfun(X, X)          # built once, sliced per fold
                        if not np.all(np.isfinite(K_full)):
                            continue
                        for a in ALPHAS:
                            cv = 0.0
                            try:
                                for tr, va in kf:
                                    coef = krr(K_full[np.ix_(tr, tr)], y[tr], a)
                                    cv += np.sqrt(np.mean(
                                        (y[va] - K_full[np.ix_(va, tr)] @ coef) ** 2))
                            except np.linalg.LinAlgError:
                                continue
                            cv /= len(kf)
                            if np.isfinite(cv) and cv < bv:
                                bv, best = cv, (a, kfun, pval)
                    if best is None:                 # every candidate failed
                        errs[kname].append(np.nan); sel[kname].append((np.nan, np.nan))
                        continue
                    a, kfun, pval = best             # refit on the full sample
                    coef = krr(kfun(X, X), y, a)
                    errs[kname].append(np.sqrt(np.mean((y_te - kfun(X_te, X) @ coef) ** 2)))
                    sel[kname].append((pval, a))
            for k in KERNELS:
                results[k][N] = errs[k]; chosen[k][N] = sel[k]
            print(f"[P={P} {f_type}] N={N}: " + ", ".join(
                f"{k}={np.nanmedian(errs[k]):.2e}"
                f"(p={np.median([t[0] for t in sel[k]]):g})" for k in KERNELS), flush=True)
        np.savez(f"scalar_P{P}_{f_type}.npz",
                 **{f"{k}_{N}": np.array(results[k][N]) for k in KERNELS for N in Ns},
                 **{f"sel_{k}_{N}": np.array(chosen[k][N]) for k in KERNELS for N in Ns})
    print(f"P={P} done", flush=True)


run_scalar(1)
run_scalar(5)

In [ ]:
# ---- Figure 1: four panels, median + IQR tubes, rates fitted on the medians ----
import matplotlib.pyplot as plt
from scipy.stats import linregress

NS = np.array(Ns)
COL = {"Matern_1.5": ("Matérn 1.5", "tab:blue"), "Matern_2.5": ("Matérn 2.5", "orange"),
       "RBF": ("RBF", "green"), "Szego_prod": ("Szegö", "red")}

for P in [1, 5]:
    for f_type in ["exp", "poly"]:
        z = np.load(f"scalar_P{P}_{f_type}.npz")
        plt.figure(figsize=(8.5, 5))
        for key, (name, c) in COL.items():
            data = [z[f"{key}_{N}"] for N in NS]
            med = np.array([np.nanmedian(d) for d in data])
            q1 = np.array([np.nanpercentile(d, 25) for d in data])
            q3 = np.array([np.nanpercentile(d, 75) for d in data])
            slope, *_ = linregress(np.log(NS), np.log(med))
            plt.plot(NS, med, marker='o', ms=3, lw=1.6, color=c,
                     label=f"{name}: N^({slope:.2f})")
            plt.fill_between(NS, q1, q3, color=c, alpha=0.18)
        plt.xscale('log'); plt.yscale('log')
        plt.xlabel("N"); plt.ylabel("Test Error")
        plt.grid(True, which="both", ls=":", lw=0.5)
        plt.legend(fontsize=9, title="Kernels")
        plt.tight_layout()
        plt.savefig(f"P={P}-f={f_type}-krr-tube{FIG_SUFFIX}.pdf")
        plt.show()

In [ ]:
# ---- boundary check: how often does CV select an endpoint of the shared grid? ----
for P in [1, 5]:
    for f_type in ["exp", "poly"]:
        z = np.load(f"scalar_P{P}_{f_type}.npz")
        for key in KERNELS:
            grid = C_GRID if key == "Szego_prod" else S_GRID
            sel = np.concatenate([z[f"sel_{key}_{N}"][:, 0] for N in Ns])
            big = np.concatenate([z[f"sel_{key}_{N}"][:, 0] for N in Ns if N >= 700])
            print(f"P={P} {f_type:>4} {key:<11} all N: "
                  f"{np.mean(np.isclose(sel, min(grid))):>4.0%} at min, "
                  f"{np.mean(np.isclose(sel, max(grid))):>4.0%} at max   |   "
                  f"N>=700: {np.mean(np.isclose(big, min(grid))):>4.0%} / "
                  f"{np.mean(np.isclose(big, max(grid))):>4.0%}")